In [1]:
import numpy as np

# Van Magnan | Homework 3 | Math 540
****
## Problem 4 
### part c

In [2]:
A_inv = np.array([[-998,999],[999,-1000]])
A_inv

array([[ -998,   999],
       [  999, -1000]])

In [3]:
b_hat = np.array([1998.99,1997.01])
b_hat

array([1998.99, 1997.01])

In [4]:
x_hat = A_inv.dot(b_hat)
x_hat

array([ 20.97, -18.99])

### part d

In [5]:
x = np.array([1,1])
b = np.array([1999,1997])

In [6]:
np.linalg.norm(x-x_hat, ord=1)/np.linalg.norm(x, ord=1)

19.979999999981374

In [7]:
(1999**2)*np.linalg.norm(b_hat-b, ord=1)/np.linalg.norm(b, ord=1)

20.000005004986818

These are pretty close.

****
## Problem 6

In [19]:
n = 60
A = np.eye(n,n) - np.tril(np.ones((n,n)),-1)
A[:,n-1] = 1

In [20]:
np.linalg.cond(A)

26.803535522538088

In [21]:
b = np.random.randn(n)
x = np.linalg.solve(A,b) # I think there was a typo in the HW assignment here

In [22]:
r = A.dot(x)-b
relRes = np.linalg.norm(r)/(np.linalg.norm(A)*np.linalg.norm(x))

In [23]:
relError = np.linalg.cond(A)*relRes

In [24]:
relError

0.42400254321859493

In [25]:
# apply iterative refinement

d = np.linalg.solve(A,r)

y = x-d

In [26]:
new_r = A.dot(y)-b

In [27]:
new_relRes = np.linalg.norm(new_r)/(np.linalg.norm(A)*np.linalg.norm(y))
new_relError = np.linalg.cond(A)*new_relRes

In [28]:
new_relError

1.9787686510065537e-16

Interesting- this only took one iteration to resolve our issue, but the name suggests that one may have to repeat this multiple times (i.e. iterate) to get an accurate solution. Would one expect more iterations be necessary for ill-conditioned $A$, perhaps?
****
## Problem 7
### part b (and c)

In [35]:


n_list = [7,9,11,13]
coeff_list = []
cond_list = []

for n in n_list:
    x = np.zeros(n+1)
    for i in range(n+1):
        x[i] = 1+i/n
    A = np.zeros((n+1,n+1))
    
    
    for i in range(n+1):
        for j in range(n+1):
            A[i,j] = x[i]**j
    
    #calculate condition number
    cond_list.append(np.linalg.cond(A))
    
    
    # find F vector
    f = A.dot(np.ones(n+1))
              
    coeffs = np.linalg.solve(A,f)
    
    coeff_list.append(coeffs)

In [36]:
for i in range(4):
    print('The coefficients for n = ',  n_list[i],  ' are: ',  coeff_list[i] )
    print('')

The coefficients for n =  7  are:  [1. 1. 1. 1. 1. 1. 1. 1.]

The coefficients for n =  9  are:  [1.00000002 0.99999987 1.00000045 0.99999914 1.00000103 0.9999992
 1.00000041 0.99999987 1.00000002 1.        ]

The coefficients for n =  11  are:  [0.99997545 1.00018923 0.99934055 1.00137133 0.99810923 1.00181501
 0.99876217 1.0005998  0.99979762 1.00004529 0.99999395 1.00000037]

The coefficients for n =  13  are:  [ 1.0120563   0.8898833   1.46228283 -0.18113349  3.04920241 -1.54940918
  3.33995571 -0.60433712  1.82170441  0.68948309  1.08415581  0.98450962
  1.00173574  0.99991058]



### part c

In [37]:
for i in range(4):
    print('The condition number for n = ',  n_list[i],  ' is: ',  cond_list[i] )
    print('')

The condition number for n =  7  is:  468856472.9267425

The condition number for n =  9  is:  267610688021.69016

The condition number for n =  11  is:  160327117995054.16

The condition number for n =  13  is:  9.933552124266405e+16



### part d

In [46]:
# note: this is not the most efficient way to have structured this problem  
# in hindsight, as I am duplicating a lot of this code. Oops

error_list = []

for n in n_list:
    x = np.zeros(n+1)
    for i in range(n+1):
        x[i] = 1+i/n
    A = np.zeros((n+1,n+1))
    
    for i in range(n+1):
        for j in range(n+1):
            A[i,j] = x[i]**j
            
    f = A.dot(np.ones(n+1))        
    coeffs = np.linalg.solve(A,f)
    
    r = A.dot(coeffs) - f
    
    relRes = np.linalg.norm(r)/np.linalg.norm(f)
    
    error_list.append(np.linalg.cond(A)*relRes)

In [47]:
for i in range(4):
    print('The relative error for n = ',  n_list[i],  ' is: ',  error_list[i] )
    print('')

The relative error for n =  7  is:  2.4357990418592296e-08

The relative error for n =  9  is:  2.1918624850346604e-05

The relative error for n =  11  is:  0.017619773610668802

The relative error for n =  13  is:  7.612308748504223



We see that as $n$ increases by $2$, the relative error is increasing by a factor of about  $10^3$.

****
## Problem 8

In [ ]:
def con_bidiag(a,b):
    
    # given the diagonal a, and the superdiagonal b
    # of a bidiagonal matrix B, this function computes
    # the condition number (in the infinity norm)
    # of the matrix B in O(n) operations
    
    n = len(a)
    # norm of b
    normB = np.max(np.abs(a)+np.abs(np.append(b,0)))
    
    row_norms = np.zeros(n)
    row_norms[n-1] = 1/np.abs(a[n-1])
    
    for i in range(n-2,-1,-1):
        row_norms[i] = (1/np.abs(a[i]))*(1+np.abs(b[i])*row_norms[i+1])
        
    normBinv = np.max(row_norms)
    
    return normB*normBinv